# Homework 3: Building an NDArray library

In this homework, you will build a simple backing library for the processing that underlies most deep learning systems: the n-dimensional array (a.k.a. the NDArray).  Up until now, you have largely been using numpy for this purpose, but this homework will walk you through developing what amounts to your own (albeit much more limited) variant of numpy, which will support both CPU and GPU backends.  What's more, unlike numpy (and even variants like PyTorch), you won't simply call out to existing highly-optimized variants of matrix multiplication or other manipulation code, but actually write your own versions that are reasonably competitive will the highly optimized code backing these standard libraries (by some measure, i.e., "only 2-3x slower" ... which is a whole lot better than naive code that can easily be 100x slower).  This class will ultimately be integrated into `needle`, but for this assignment you can _only_ focus on the ndarray module, as this will be the only subject of the tests.

**Note**: To avoid exhausting limited GPU resources in Colab, start by using CPU runtime for coding and testing non-GPU functions. Switch to GPU runtime when testing CUDA or GPU-accelerated code. This approach ensures efficient GPU usage and prevents running out of resources during critical tasks.


# 作业3：构建一个NDArray库

在本作业中，你将构建一个简单的底层库，用于支撑大多数深度学习系统背后的核心处理：n维数组（即NDArray）。到目前为止，你大多是在使用numpy来完成这一目的，但本作业将引导你开发出相当于你自己的（尽管功能受限得多）numpy变体，该变体将同时支持CPU和GPU后端。更重要的是，与numpy（甚至包括PyTorch等变体）不同，你不会简单地调用现成的高度优化的矩阵乘法或其他操作代码，而是需要自己编写具有一定竞争力的版本，与这些标准库背后高度优化的代码相比（从某种衡量标准来看，即“仅慢2-3倍”……这远比那些可能轻易慢上100倍的朴素代码要好得多）。这个类最终将被集成到`needle`中，但对于本次作业，你_只能_专注于ndarray模块，因为这将只是测试的唯一主题。

**注意**：为避免耗尽Colab中有限的GPU资源，请先使用CPU运行时来编写和测试非GPU函数。在测试CUDA或GPU加速代码时，再切换到GPU运行时。这种方法可以确保高效使用GPU，并防止在关键任务期间耗尽资源。

In [ ]:
# Code to set up the assignment
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/
!mkdir -p 10714
%cd /content/drive/MyDrive/10714
!git clone https://github.com/dlsys10714/hw3.git
%cd /content/drive/MyDrive/10714/hw3

!pip3 install --upgrade --no-deps git+https://github.com/dlsys10714/mugrade.git
!pip3 install pybind11

In [ ]:
# REQUIRED FOR MUGRADE
MY_API_KEY = "<FILL YOUR API KEY HERE>"
HW3_NAME = "hw3"

In [ ]:
!make

The make command reads the Makefile in the current directory. The Makefile contains rules that define how to build targets (like executables or libraries). For each target specified in the Makefile, make checks the timestamps of the target file and its dependencies (like .c, .cpp, or .h files). If any dependency has been modified recently, it must rebuild the target.

`make` 命令会读取当前目录下的 Makefile 文件。Makefile 中包含了一系列规则，这些规则定义了如何构建目标（例如可执行文件或库）。对于 Makefile 中指定的每个目标，`make` 会检查目标文件及其依赖项（如 `.c`、`.cpp` 或 `.h` 文件）的时间戳。如果任一依赖项在最近被修改过，`make` 就会重新构建该目标。

In [ ]:
%set_env PYTHONPATH ./python
%set_env NEEDLE_BACKEND nd

In [ ]:
import sys
sys.path.append('./python')

## Getting familiar with the NDArray class

As you get started with this homework, you should first familiarize yourself with the `NDArray.py` class we have provided as a starting point for the assignment.  The code is fairly brief (it's ~500 lines, but a lot of these are comments provided for the functions you'll implement).

At its core, the NDArray class is a Python wrapper for handling operations on generic n-dimensional arrays.  Recall that virtually any such array will be stored internally as a vector of floating point values, i.e.,

```c++
float data[size];
```

and then the actual access to different dimensions of the array are all handled by additional fields (such as the array shape, strides, etc) that indicates how this "flat" array maps to n-dimensional structure.  In order to achieve any sort of reasonable speed, the "raw" operations (like adding, binary operations, but also more structured operations like matrix multiplication, etc), all need to be written at some level in some native language like C++ (including e.g., making CUDA calls).  But a large number of operations likes transposing, broadcasting, sub-setting of matrices, and other, can all be handled by just adjusting the high-level structure of the array, like it's strides.

The philosophy behind the NDArray class is that we want _all_ the logic for handling this structure of the array to be written in Python.  Only the "true" low level code that actually performs the raw underlying operations on the flat vector (as well as the code to manage these flat vectors, as they might need to e.g., be allocated on GPUs), is written in C++.  The precise nature of this separation will likely start to make more sense to you as you work through the assignment, but generally speaking everything that can be done in Python, is done in Python; often e.g., at the cost of some inefficiencies ... we call `.compact()` (which copies memory) liberally in order to make the underlying C++ implementations simpler.

In more detail, there are five fields within the NDArray class that you'll need to be familiar with (note that the real class member these all these fields is preceded by an underscore, e.g., `_handle`, `_strides`, etc, some of which are then exposed as a public property ... for all your code it's fine to use the internal, underscored version).

1. `device` - A object of type `BackendDevice`, which is a simple wrapper that contains a link to the underlying device backend (e.g., CPU or CUDA).
2. `handle` - A class objected that stores the underlying memory of the array.  This is allocated as a class of type `device.Array()`, though this allocation all happens in the provided code (specifically the `NDArray.make` function), and you don't need to worry about calling it yourself.
3. `shape` - A tuple specifying the size of each dimension in the array.
4. `strides` - A tuple specifying the strides of each dimension in the array.
5. `offset` - An integer indicating where in the underlying `device.Array` memory the array actually starts (it's convenient to store this so we can more easily manage pointing back to existing memory, without having to track allocations).

By manipulating these fields, even pure Python code can perform a lot of the needed operations on the array, such as permuting the dimensions (i.e., transposing), broadcasting, and more.  And then for the raw array manipulation calls, the `device` class has a number of methods (implemented in C++) that contains the necessary implementations.

There are a few points to note:

* Internally, the class can use _any_ efficient means of operating on arrays of data as a "device" backend.  Even, for example, a numpy array, but where instead of actually using the `numpy.ndarray` to represent the n-dimensional array, we just represent a "flat" 1D array in numpy, then call the relevant numpy methods to implement all the needed operators on this raw memory.  This is precisely what we do in the `ndarray_backend_numpy.py` file, which essentially provided a "stub reference" that just uses numpy for everything.  You can use this class to help you  better debug your own "real" implementations for the "native" CPU and GPU backends.
* Of particular importance for many of your Python implementations will be the `NDArray.make` call:
```python
def make(shape, strides=None, device=None, handle=None, offset=0):
```
which creates a new NDArray with the given shape, strides, device, handle, and offset.  If `handle` is not specified (i.e., no pre-existing memory is referenced), then the call will allocate the needed memory, but if handle _is_ specified then no new memory is allocated, but the new NDArray points the same memory as the old one.  It is important to efficient implementations that as many of your functions as possible _don't_ allocate new memory, so you will want to use this call in many cases to accomplish this.
* The NDArray has a `.numpy()` call that converts the array to numpy.  This is _not_ the same as the "numpy_device" backend: this creates an actual `numpy.ndarray` that is equivalent to the given NDArray, i.e., the same dimensions, shape, etc, though not necessarily the same strides (Pybind11 will reallocate memory for matrices that are returned in this manner, which can change the striding).


## 熟悉 NDArray 类

在开始本次作业之前，你应该先熟悉我们提供的 `NDArray.py` 类，这是本次作业的起点。代码相当简短（大约 500 行，但其中很多是为你将要实现的函数提供的注释）。

从核心上讲，NDArray 类是一个 Python 包装器，用于处理通用的 n 维数组操作。回想一下，几乎任何这样的数组在内部都会存储为一个浮点值向量，即：

```c++
float data[size];
```

然后，对数组不同维度的实际访问都由额外的字段（如数组形状、步长等）来处理，这些字段指示了这个“扁平”数组如何映射到 n 维结构。为了达到任何合理的速度，“原始”操作（如加法、二元运算，但也包括更结构化的操作，如矩阵乘法等）都需要在某种层面上用 C++ 等原生语言编写（包括例如调用 CUDA）。但大量的操作，如转置、广播、矩阵子集提取等，都可以通过仅仅调整数组的高层结构（比如它的步长）来处理。

NDArray 类背后的理念是，我们希望所有处理数组结构的逻辑都用 Python 编写。只有“真正”的低层代码——即实际在扁平向量上执行原始底层操作的代码（以及管理这些扁平向量的代码，因为它们可能需要例如在 GPU 上分配）——才用 C++ 编写。这种分离的确切性质可能会在你完成作业的过程中逐渐变得清晰，但总的来说，凡是能用 Python 完成的事情，都用 Python 完成；通常，例如，以牺牲一些效率为代价……我们会大量调用 `.compact()`（它会复制内存），以使底层的 C++ 实现更简单。

更详细地说，NDArray 类中有五个字段你需要熟悉（注意，这些字段的实际类成员名前都有下划线，例如 `_handle`、`_strides` 等，其中一些随后作为公共属性暴露出来……对于你的所有代码，使用内部的带下划线版本即可）。

1. `device` - 一个 `BackendDevice` 类型的对象，它是一个简单的包装器，包含对底层设备后端（例如 CPU 或 CUDA）的链接。
2. `handle` - 一个类对象，存储数组的底层内存。它被分配为 `device.Array()` 类型的类，不过这种分配都发生在提供的代码中（特别是 `NDArray.make` 函数），你不需要担心自己调用它。
3. `shape` - 一个元组，指定数组中每个维度的大小。
4. `strides` - 一个元组，指定数组中每个维度的步长。
5. `offset` - 一个整数，指示数组在底层 `device.Array` 内存中的实际起始位置（存储这个值很方便，这样我们可以更容易地管理指向现有内存的引用，而不必跟踪分配）。

通过操作这些字段，即使是纯 Python 代码也能执行数组上的许多所需操作，例如置换维度（即转置）、广播等。然后，对于原始数组操作调用，`device` 类有许多方法（用 C++ 实现），包含必要的实现。

有几点需要注意：

* 在内部，该类可以使用任何高效的方式来操作数据数组作为“设备”后端。例如，甚至可以使用 numpy 数组，但不是实际使用 `numpy.ndarray` 来表示 n 维数组，而是仅在 numpy 中表示一个“扁平”的 1D 数组，然后调用相关的 numpy 方法在这个原始内存上实现所有需要的运算符。这正是我们在 `ndarray_backend_numpy.py` 文件中所做的，它本质上提供了一个“存根参考”，只是用 numpy 处理一切。你可以使用这个类来帮助你更好地调试你自己为“原生”CPU 和 GPU 后端编写的“真实”实现。
* 对你的许多 Python 实现来说，特别重要的是 `NDArray.make` 调用：
```python
def make(shape, strides=None, device=None, handle=None, offset=0):
```
它创建一个具有给定形状、步长、设备、句柄和偏移量的新 NDArray。如果未指定 `handle`（即没有引用预先存在的内存），则该调用将分配所需的内存；但如果指定了 `handle`，则不会分配新内存，但新的 NDArray 指向与旧数组相同的内存。对于高效实现来说，尽可能多的函数 _不要_ 分配新内存非常重要，因此在许多情况下你会希望使用这个调用来实现这一点。
* NDArray 有一个 `.numpy()` 调用，可以将数组转换为 numpy。这 _不是_ 与“numpy_device”后端相同：这会创建一个实际的 `numpy.ndarray`，它与给定的 NDArray 等价，即具有相同的维度、形状等，但步长不一定相同（Pybind11 会为以这种方式返回的矩阵重新分配内存，这可能会改变步长）。

## Part 1: Python array operations

As a starting point for your class, implement the following functions in the `ndarray.py` file:

- `reshape()`
- `permute()`
- `broadcast_to()`
- `__getitem__()`

The inputs/outputs of these functions are all described in the docstring of the function stub.  It's important to emphasize that _none_ of these functions should reallocate memory, but should instead return NDArrays that share the same memory with `self`, and just use clever manipulation of shape/strides/etc in order to obtain the necessary transformations.

To get started you can refer to the hints mentioned in the class slides for transpose, broadcast and slicing operator.

One thing to note is that the `__getitem__()` call, unlike numpy, will never change the number of dimensions in the array.  So e.g., for a 2D NDArray `A`, `A[2,2]` would return a still-2D with one row and one column.  And e.g. `A[:4,2]` would return a 2D NDarray with 4 rows and 1 column.

You can rely on the `ndarray_backend_numpy.py` module for all the code in this section.  You can also look at the results of equivalent numpy operations (the test cases should illustrate what these are).

After implementing these functions, you should pass/submit the following tests.  Note that we test all of these four functions within the test below, and you can incrementally try to pass more tests as you implement each additional function.

## 第一部分：Python 数组操作

作为课程的起点，请在 `ndarray.py` 文件中实现以下函数：

- `reshape()`
- `permute()`
- `broadcast_to()`
- `__getitem__()`

这些函数的输入/输出都在函数桩的文档字符串中进行了描述。需要强调的是，这些函数都_不应_重新分配内存，而应返回与 `self` 共享相同内存的 NDArray，仅通过巧妙地操作形状/步长等来实现所需的变换。

首先，您可以参考课堂幻灯片中关于转置、广播和切片操作符的提示。

需要注意的一点是，与 numpy 不同，`__getitem__()` 调用永远不会改变数组的维度数量。例如，对于二维 NDArray `A`，`A[2,2]` 将返回一个仍为二维、只有一行一列的数组。再如，`A[:4,2]` 将返回一个具有 4 行 1 列的二维 NDArray。

您可以依赖 `ndarray_backend_numpy.py` 模块来完成本节中的所有代码。您也可以查看等效 numpy 操作的结果（测试用例应该能说明这些操作是什么）。

实现这些函数后，您应该通过/提交以下测试。请注意，我们在下面的测试中测试了这四个函数，您可以在实现每个附加函数时逐步尝试通过更多测试。

In [ ]:
!python3 -m pytest -v -k "(permute or reshape or broadcast or getitem) and cpu and not compact"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_python_ops"

## Part 2: CPU Backend - Compact and setitem

Implement the following functions in `ndarray_backend_cpu.cc`:
* `Compact()`
* `EwiseSetitem()`
* `ScalarSetitem()`

To see why these are all in the same category, let's consider the implementation of the `Compact()` function.  Recall that a matrix is considered compact if it is layed out sequentially in memory in "row-major" form (but really a generalization of row-many to higher dimensional arrays), i.e. with the last dimension first, followed by the second to last dimension, etc, all the way to the first.  In our implementation, we also require that the total size of allocated backend array be equal to the size of the array (i.e., the underlying array also can't have any data before or after the array data, which e.g., implies that the `.offset` field equals zero).

Now let's consider, using a 3D array as a an example, of how a compact call might work.  Here `shape` and `strides` are the shape and strides of the matrix being compacted (i.e., before we have compacted it).

```c++
cnt = 0;
for (size_t i = 0; i < shape[0]; i++)
    for (size_t j = 0; j < shape[1]; j++)
        for (size_t k = 0; k < shape[2]; k++)
            out[cnt++] = in[strides[0]*i + strides[1]*j + strides[2]*k];
```
In other words, we're converting from a stride-based representation of the matrix to a purely sequential one.

Now, the challenge in implementing `Compact()` is that you want the method to work for any number of input dimensions.  It's easy to specialize for different fixed-dimension-size arrays, but for a generic implementation, you'll want to think about how to do this same operation where you effectively want a "variable number of for loops".  As a hint, one way to do this is to maintain a vector of indices (of size equal to the number of dimensions), and then manually increment them in a loop (including a "carry" operation when any of the reaches their maximum size).

However, if you get really stuck with this, you can alway use the fact that we're probably not going to ask you to deal with matrices of more than 6 dimensions (though we _will_ use 6 dimensions, for the im2col operation we discussed in class).


#### The connection to setitem
The setitem functionality, while seemingly quite different, is actually intimately related to `Compact()`.  `__setitem()__` is the Python function called when setting some elements of the object, i.e.,
```python
A[::2,4:5,9] = 0 # or = some_other_array
```
How would you go about implementing this?  In the `__getitem()__` call above, you already implemented a method to take a subset of a matrix without copying (but just modifying strides).  But how would you actually go about _setting_ elements of this array?  In virtually all the other settings in this homework, we call `.compact()` before setting items in an output array, but in this case it doesn't work: calling `.compact()` would copy the subset array to some new memory, but the whole point of the `__setitem__()` call is that we want to modify existing memory.

If you think about this for a while, you'll realize that the answer looks a lot like `.compact()` but in reverse.  If we want to assign a (itself already compact) right hand side matrix to a `__getitem()__` results, then we need to here like `shape` and `stride` be those fields of the _output_ matrix.  Then we could implement the setitem call as follows

```c++
cnt = 0;
for (size_t i = 0; i < shape[0]; i++)
    for (size_t j = 0; j < shape[1]; j++)
        for (size_t k = 0; k < shape[2]; k++)
            out[strides[0]*i + strides[1]*j + strides[2]*k] = in[cnt++]; // or "= val;"
```
Due to this similarity, if you implement your indexing strategy in a modular fashion, you'll be able to reuse it between the `Compact()` call and the `EwiseSetitem()` and `ScalarSetitem()` calls.

**Note**: Don't forget to run make before executing the tests to rebuild your modified C++ code.

In [ ]:
!python3 -m pytest -v -k "(compact or setitem) and cpu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_compact_setitem"

## Part 3: CPU Backend - Elementwise and scalar operations

Implement the following functions in `ndarray_backend_cpu.cc`:

* `EwiseMul()`, `ScalarMul()`
* `EwiseDiv()`, `ScalarDiv()`
* `ScalarPower()`
* `EwiseMaximum()`, `ScalarMaximum()`
* `EwiseEq()`, `ScalarEq()`
* `EwiseGe()`, `ScalarGe()`
* `EwiseLog()`
* `EwiseExp()`
* `EwiseTanh()`

You can look at the included
`EwiseAdd()` and `ScalarAdd()` functions (plus the invocations from `NDArray` in order to understand the required format of these functions.

Note that unlike the remaining functions mentioned here, we do not include function stubs for each of these functions.  This is because, while you can implement these naively just through implementing each function separately, though this will end up with a lot of duplicated code.  You're welcome to use e.g., C++ templates or macros to address this problem (but these would only be exposed internally, not to the external interface).

In [ ]:
!python3 -m pytest -v -k "(ewise_fn or ewise_max or log or exp or tanh or (scalar and not setitem)) and cpu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_ops"

## Part 4: CPU Backend - Reductions


Implement the following functions in `ndarray_backend_cpu.cc`:

* `ReduceMax()`
* `ReduceSum()`

In general, the reduction functions `.max()` and `.sum()` in NDArray take the max or sum across a specified axis specified by the `axis` argument (or across the entire array when `axis=None`); note that we don't support axis being a set of axes, though this wouldn't be too hard to add if you desired (but it's not in the interface you should implement for the homework).

Because summing over individual axes can be a bit tricky, even for compact arrays, these functions (in Python) in Python simplify things by permuting the last axis to the be the one reduced over (this is what the `reduce_view_out()` function in NDArray does), then compacting the array.  So for your `ReduceMax()` and `ReduceSum()` functions you implement in C++, you can assume that both the input and output arrays are contiguous in memory, and you want to just reduce over contiguous elements of size `reduce_size` as passed to the C++ functions. 

In [ ]:
!python3 -m pytest -v -k "reduce and cpu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_reductions"

## Part 5: CPU Backend - Matrix multiplication

Implement the following functions in `ndarray_backend_cpu.cc`:

* `Matmul()`
* `MatmulTiled()`
* `AlignedDot()`

The first implementation, `Matmul()` can use the naive three-nested-for-loops algorithm for matrix multiplication.  However, the `MatmulTiled()` performs the same matrix multiplication on memory laid out in tiled form, i.e., as a contiguous 4D array
```c++
float[M/TILE][N/TILE][TILE][TILE];
```

Make sure to initialize the output matrix to zero before populating it with the correct values during matrix multiplication.

Note that the Python `__matmul__` code already does the conversion to tiled form when all sizes of the matrix multiplication are divisible by `TILE`, so your code just needs to implement the multiplication in this form.  In order to make the methods efficient, you will want to make use of (after you implement it), the `AlignedDot()` function, which will enable the compiler to efficiently make use of vector operations and proper caching.  The output matrix will also be in the tiled form above, and the Python backend will take care of the conversion to a normal 2D array.

Note that in order to get the most speedup possible from you tiled version, you may want to use the clang compiler with colab instead of gcc.  To do this, run the following command before building your code.

In [ ]:
!python3 -m pytest -v -k "matmul and cpu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_matmul"

In [ ]:
!export CXX=/usr/bin/clang++ && make

## Part 6: CUDA Backend - Compact and setitem

Implement the following functions in `ndarray_backend_cuda.cu`:
* `Compact()`
* `EwiseSetitem()`
* `ScalarSetitem()`


For this portion, you'll implement the compact and setitem calls in the CUDA backend.  This is fairly similar to the C++ version, however, depending on how you implemented that function, there could also be some substantial differences.  We specifically want to highlight a few differences between the C++ and the CUDA implementations, however.

First, as with the example functions implemented in the CUDA backend code, for all the functions above you will actually want to implement two functions: the basic functions listed above that you will call from Python, and the corresponding CUDA kernels that will actually perform the computation.  For the most part, we only provide the prototype for the "base" function in the `ndarray_backend_cuda.cu` file, and you will need to define and implement the kernel function yourself.  However, to see how these work, for the `Compact()` call we are providing you with the _complete_ `Compact()` call, and the function prototype for the `CompactKernel()` call.

One thing you may notice is the seemingly odd use of a `CudaVec` struct, which is a struct used to pass shape/stride parameters.  In the C++ version we used the STL `std::vector` variables to store these inputs (and the same is done in the base `Compact()` call, but CUDA kernels cannot operation on STL vectors, so something else is needed).  Furthermore, although we _could_ convert the vectors to normal CUDA arrays, this would be rather cumbersome, as we would need to call `cudaMalloc()`, pass the parameters as integer pointers, then free them after the calls.  Of course such memory management is needed for the actual underlying data in the array, but it seems like overkill to do it for just passing a variable-sized small tuple of shape/stride values.  The solution is to create a struct that has a "maximize" size for the number of dimensions an array can have, and then just store the actual shape/stride data in the first entries of these fields.  This is all done by the included `CudaVec` struct and `VecToCuda()` function, and you can just use these as provided for all the CUDA kernels that require passing shape/strides to the kernel itself.

The other (more conceptual) big difference between the C++ and CUDA implementations of `Compact()` is that in C++ you will typically loop over the elements of the non-compact array sequentially, which allows you to perform some optimizations with respect to computing the corresponding indices between the compact and non-compact arrays.  In CUDA, you cannot do this, and will need to implement code that can directly map from an index in the compact array to one in the strided array.

As before, we recommend you implement your code in such as way that it can easily be re-used between the `Compact()`, and `Setitem()` calls.  As a short note, remember that if you want to call a (separate, non-kernel) function from kernel code, you need to define it as a `__device__` function. In CUDA, `__global__` is used to define a function that runs on the GPU and is called from the CPU host, while `__device__` defines a function that runs and is called only on the GPU from other GPU function code


## 第6部分：CUDA 后端——Compact 与 setitem

在 `ndarray_backend_cuda.cu` 中实现以下函数：
* `Compact()`
* `EwiseSetitem()`
* `ScalarSetitem()`


在这一部分中，你将在 CUDA 后端实现 compact 和 setitem 调用。这与 C++ 版本相当类似，但根据你之前实现该函数的方式，也可能存在一些实质性差异。不过，我们特别想强调 C++ 与 CUDA 实现之间的几个不同点。

首先，与 CUDA 后端代码中已实现的示例函数一样，对于上述所有函数，你实际上需要实现两个函数：上面列出的、将从 Python 调用的基础函数，以及实际执行计算的对应 CUDA 内核。在大多数情况下，我们只在 `ndarray_backend_cuda.cu` 文件中提供了“基础”函数的原型，你需要自行定义并实现内核函数。不过，为了了解其工作方式，对于 `Compact()` 调用，我们为你提供了完整的 `Compact()` 调用，以及 `CompactKernel()` 调用的函数原型。

你可能会注意到一个看似奇怪的 `CudaVec` 结构体的使用，这是一个用于传递形状/步幅参数的结构体。在 C++ 版本中，我们使用 STL 的 `std::vector` 变量来存储这些输入（基础 `Compact()` 调用中也是如此），但 CUDA 内核无法操作 STL 向量，因此需要其他方式。此外，虽然我们可以将向量转换为普通的 CUDA 数组，但这会相当繁琐，因为我们需要调用 `cudaMalloc()`，将参数作为整数指针传递，然后在调用后释放它们。当然，对于数组中的实际底层数据，这种内存管理是必要的，但仅仅为了传递一个大小可变的形状/步幅小元组而这样做似乎有些小题大做。解决方案是创建一个结构体，其维度数量具有“最大”大小，然后将实际的形状/步幅数据存储在这些字段的前几个条目中。这一切都由附带的 `CudaVec` 结构体和 `VecToCuda()` 函数完成，你可以直接使用它们来传递给所有需要将形状/步幅传入内核的 CUDA 内核。

另一个（更具概念性的）C++ 与 CUDA 实现 `Compact()` 之间的重大区别在于，在 C++ 中，你通常会顺序遍历非紧凑数组的元素，这使你可以针对紧凑数组与非紧凑数组之间对应索引的计算进行一些优化。而在 CUDA 中，你无法这样做，你需要实现能够直接将紧凑数组中的索引映射到带步幅数组中的索引的代码。

和之前一样，我们建议你以易于在 `Compact()` 和 `Setitem()` 调用之间复用代码的方式来实现。简单提醒一下，如果你想从内核代码中调用（单独的、非内核的）函数，你需要将其定义为 `__device__` 函数。在 CUDA 中，`__global__` 用于定义在 GPU 上运行并由 CPU 主机调用的函数，而 `__device__` 则定义仅在 GPU 上运行、且只能由其他 GPU 函数代码调用的函数。

In [ ]:
!python3 -m pytest -v -k "(compact or setitem) and cuda"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_compact_setitem"

## Part 7: CUDA Backend - Elementwise and scalar operations

Implement the following functions in `ndarray_backend_cuda.cu`:

* `EwiseMul()`, `ScalarMul()`
* `EwiseDiv()`, `ScalarDiv()`
* `ScalarPower()`
* `EwiseMaximum()`, `ScalarMaximum()`
* `EwiseEq()`, `ScalarEq()`
* `EwiseGe()`, `ScalarGe()`
* `EwiseLog()`
* `EwiseExp()`
* `EwiseTanh()`

Again, we don't provide these function prototypes, and you're welcome to use C++ templates or macros to make this implementation more compact.  You will also want to uncomment the appropriate regions of the Pybind11 code once you've implemented each function.

In [2]:
!python3 -m pytest -v -k "(ewise_fn or ewise_max or log or exp or tanh or (scalar and not setitem)) and cuda"

============================= test session starts ==============================
platform linux -- Python 3.10.12, pytest-9.1.1, pluggy-1.6.0 -- /home/k/Projects/my_dlsys/.venv/bin/python3
cachedir: .pytest_cache
rootdir: /home/k/Projects/my_dlsys/hw3
collected 136 items / 113 deselected / 23 selected                             

tests/hw3/test_ndarray.py::test_ewise_fn[cuda-shape0-multiply] PASSED    [  4%]
tests/hw3/test_ndarray.py::test_ewise_fn[cuda-shape0-divide] PASSED      [  8%]
tests/hw3/test_ndarray.py::test_ewise_fn[cuda-shape0-add] PASSED         [ 13%]
tests/hw3/test_ndarray.py::test_ewise_fn[cuda-shape0-subtract] PASSED    [ 17%]
tests/hw3/test_ndarray.py::test_ewise_fn[cuda-shape0-equal] PASSED       [ 21%]
tests/hw3/test_ndarray.py::test_ewise_fn[cuda-shape0-greater_than] PASSED [ 26%]
tests/hw3/test_ndarray.py::test_ewise_fn[cuda-shape1-multiply] PASSED    [ 30%]
tests/hw3/test_ndarray.py::test_ewise_fn[cuda-shape1-divide] PASSED      [ 34%]
tests/hw3/test_ndarray.py:

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_ops"

## Part 8: CUDA Backend - Reductions


Implement the following functions in `ndarray_backend_cuda.cu`:

* `ReduceMax()`
* `ReduceSum()`

You can take a fairly simplistic approach here, and just use a separate CUDA thread for each individual reduction item: i.e., if there is a 100 x 20 array you are reducing over the second dimension, you could have 100 threads, each of which individually processed its own 20-dimensional array..  This is particularly inefficient for the `.max(axis=None)` calls, but we won't worry about this for the time being.  If you want a more industrial-grade implementation, you use a hierarchical mechanism that first aggregated across some smaller span, then had a secondary function that aggregated across _these_ reduced arrays, etc.  But this is not needed to pass the tests.

In [3]:
!python3 -m pytest -v -k "reduce and cuda"

============================= test session starts ==============================
platform linux -- Python 3.10.12, pytest-9.1.1, pluggy-1.6.0 -- /home/k/Projects/my_dlsys/.venv/bin/python3
cachedir: .pytest_cache
rootdir: /home/k/Projects/my_dlsys/hw3
collected 0 items / 1 error                                                    

==================================== ERRORS ====================================
__________________ ERROR collecting tests/hw3/test_ndarray.py __________________
ImportError while importing test module '/home/k/Projects/my_dlsys/hw3/tests/hw3/test_ndarray.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.10/importlib/__init__.py:126: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
tests/hw3/test_ndarray.py:4: in <module>
    import needle as ndl
E   ModuleNotFoundError: No module named 'needle'
=========================== short test summary info ============================
ER

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_reductions"

## Part 9: CUDA Backend - Matrix multiplication

Implement the following functions in `ndarray_backend_cuda.cu`:

* `Matmul()`

Finally, as your final exercise, you'll implement matrix multiplication on the GPU.  Your implementation here can roughly follow the presentation in class.  While you can pass the tests using fairly naive code here (i.e., you could just have a separate thread for each (i,j) location in the matrix, doing the matrix multiplication efficiently (to make it actually faster than a CPU version) requires cooperative fetching and the block shared memory register tiling covered in class.  Try to implement using these methods, and see how much faster you can get your code than the C++ (or numpy) backends.


In [ ]:
!python3 -m pytest -v -k "matmul and cuda"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_matmul"